# STOR 120 Midterm 3 — Interactive Python Code & Concept Drill Notebook
**Targeted Practice for Problems 2b, 3b, 3c(ii), 4a, 4b, 5a, 5b, 6a, 6b, and 6c(ii)**

### How to Use This Notebook
1. **Run Cell 1 first** (`Shift + Enter`) to load the datasets (`demographics` and `births`), the **exact docstring-stripped reference functions** printed on your exam (`Class 32` & `Class 34`), and the self-grading `check_*()` functions. *(Note: This notebook includes a built-in fallback for `datascience`, so it runs in any Python/Jupyter/Colab environment!)*
2. In each exercise cell, replace every `...` blank with the correct Python expression and run the cell.
3. Each cell immediately grades your code and prints either `PASSED!` or a specific hint.
4. Try completing the notebook once looking at the hints, then **Reset/Clear Outputs** and fill in all blanks from memory!


In [1]:
import numpy as np
from scipy import stats

# Built-in fallback so this notebook runs whether or not the `datascience` package is installed
try:
    from datascience import *
except ImportError:
    from scipy.optimize import minimize as _sp_minimize
    def make_array(*args):
        return np.array(args)
    def percentile(p, arr):
        return float(np.percentile(arr, p, method='nearest'))
    def minimize(fn):
        import inspect
        n_args = len(inspect.signature(fn).parameters)
        res = _sp_minimize(lambda params: fn(*params), np.zeros(n_args))
        return np.array(res.x)
    class Table:
        def __init__(self, data=None):
            self._data = dict(data) if data else {}
        def with_columns(self, *args):
            new_data = dict(self._data)
            for i in range(0, len(args), 2):
                new_data[args[i]] = np.asarray(args[i+1])
            return Table(new_data)
        def column(self, label):
            if isinstance(label, int):
                label = list(self._data.keys())[label]
            return self._data[label]
        @property
        def num_rows(self):
            return len(next(iter(self._data.values()))) if self._data else 0
        def sample(self, k=None, with_replacement=True):
            n = self.num_rows if k is None else k
            idx = np.random.choice(self.num_rows, size=n, replace=with_replacement)
            return Table({col: arr[idx] for col, arr in self._data.items()})

# ==============================================================================
# PROVIDED EXAM REFERENCE FUNCTIONS (DOCSTRINGS STRIPPED - CLASS 32 & CLASS 34)
# ==============================================================================
def standard_units(arr):
    return (arr - np.mean(arr)) / np.std(arr)

def correlation(t, x, y):
    return np.mean(standard_units(t.column(x)) * standard_units(t.column(y)))

def slope(t, x, y):
    r = correlation(t, x, y)
    return r * np.std(t.column(y)) / np.std(t.column(x))

def intercept(t, x, y):
    return np.mean(t.column(y)) - slope(t, x, y) * np.mean(t.column(x))

def fitted_values(t, x, y):
    return slope(t, x, y) * t.column(x) + intercept(t, x, y)

def residuals(t, x, y):
    return t.column(y) - fitted_values(t, x, y)

def prediction_at(t, x, y, new_x):
    return slope(t, x, y) * new_x + intercept(t, x, y)

def add_regression_to_table(t, x, y):
    return t.with_columns('Fitted', fitted_values(t, x, y), 'Residual', residuals(t, x, y))

# ==============================================================================
# PRE-LOADED COURSE DATASETS (demographics & births)
# ==============================================================================
np.random.seed(120)
_n_dist = 435
_college = np.round(np.random.uniform(10.0, 65.0, _n_dist), 1)
_clinton = np.round(np.clip(0.55 * _college + np.random.normal(30.0, 10.0, _n_dist), 12.0, 88.0), 1)
_income = np.round(20500 + 1250 * _college + 176 * (_clinton - 48) + np.random.normal(0, 9100, _n_dist), 0)

demographics = Table().with_columns(
    'College %', _college,
    'Clinton %', _clinton,
    'Median Income', _income
)

_n_births = 300
_gest = np.round(np.random.uniform(240, 310, _n_births), 0)
_m_age = np.round(np.random.uniform(19, 41, _n_births), 0)
_bwt = np.round(0.46 * _gest + 0.12 * _m_age - 13.5 + np.random.normal(0, 6.1, _n_births), 1)

births = Table().with_columns(
    'Gestational Days', _gest,
    'Maternal Age', _m_age,
    'Birth Weight', _bwt
)

print("Setup complete! Loaded `demographics` (435 rows) and `births` (300 rows), plus all 8 exam reference functions.")


Setup complete! Loaded `demographics` (435 rows) and `births` (300 rows), plus all 8 exam reference functions.


## Drill 1 (Fixes Problem 2b): `scipy.stats` (`cdf` vs. `ppf`) & Student's $t$-Test $p$-Values

### The 3 Rules You Need for Problem 2b:
1. **`cdf` vs. `ppf` Direction:**
   * **`stats.norm.cdf(x, loc=mean, scale=sd)`** $\rightarrow$ Takes a **horizontal-axis value `x`** and returns the **area (probability) to the LEFT of `x`**.
   * **`stats.norm.ppf(q, loc=mean, scale=sd)`** $\rightarrow$ Takes a **left-tail area/percentile `q`** ($0 < q < 1$) and returns the **horizontal-axis cutoff `x`** (inverse of `cdf`, just like `percentile`!).
2. **Degrees of Freedom (`df`) for Student's $t$-Distribution (`Class 26` & `Class 34`):**
   * **1-Sample $t$-test:** `df = n - 1`
   * **2-Sample $t$-test:** `df = min(n1, n2) - 1`
3. **The 3 Tail Rules for `stats.t.cdf(t_stat, df)`:**
   * **Left-tailed ($H_a: \mu_1 < \mu_2$):** `stats.t.cdf(t_stat, df)`
   * **Right-tailed ($H_a: \mu_1 > \mu_2$):** `1 - stats.t.cdf(t_stat, df)`
   * **Two-tailed ($H_a: \mu_1 \neq \mu_2$):** `2 * (1 - stats.t.cdf(abs(t_stat), df))` *(or `2 * stats.t.cdf(-abs(t_stat), df)`)*


In [5]:
# ==============================================================================
# DRILL 1 WORKSPACE: Fill in every `...` below and run this cell!
# ==============================================================================

# 1A (Normal CDF): Find the proportion of a Normal(mean=100, sd=15) curve to the LEFT of x = 85:
area_left_85 = stats.norm.cdf(85, loc=100, scale=15)

# 1B (Normal PPF): Find the horizontal-axis cutoff x for the 90th percentile (q = 0.90) of Normal(mean=100, sd=15):
cutoff_90th = stats.norm.ppf(0.90, loc=100, scale=15)

# 1C (Problem 2b(ii) Exact Replay): Two-sample t-test with n1 = 34, n2 = 40, and t_stat = -2.15.
# Compute the degrees of freedom `df_2b` and the TWO-TAILED p-value `pval_2b`:
n1, n2 = 34, 40
t_stat = -2.15
df_2b = ...
pval_2b = ...

# 1D (Bonus Variation): Suppose t_stat_right = 1.85 with n1 = 25, n2 = 30 for a RIGHT-TAILED test (Ha: mu1 > mu2).
df_bonus = ...
pval_right = ...

# --- AUTO-GRADER FOR DRILL 1 ---
assert not isinstance(area_left_85, type(...)) and np.isclose(area_left_85, stats.norm.cdf(85, 100, 15)), "Check 1A: stats.norm.cdf(85, loc=100, scale=15)"
assert not isinstance(cutoff_90th, type(...)) and np.isclose(cutoff_90th, stats.norm.ppf(0.90, 100, 15)), "Check 1B: stats.norm.ppf(0.90, loc=100, scale=15)"
assert df_2b == 33, f"Check 1C df: Expected min(34, 40) - 1 = 33, got {df_2b}"
assert np.isclose(pval_2b, 2 * (1 - stats.t.cdf(abs(-2.15), 33))), f"Check 1C pval_2b: Use 2 * (1 - stats.t.cdf(abs(t_stat), df_2b))"
assert df_bonus == 24 and np.isclose(pval_right, 1 - stats.t.cdf(1.85, 24)), "Check 1D: df = min(25, 30) - 1 = 24, pval = 1 - stats.t.cdf(1.85, 24)"
print("DRILL 1 PASSED! You nailed Normal CDF/PPF and Student's t-test p-values!")


AssertionError: Check 1C df: Expected min(34, 40) - 1 = 33, got Ellipsis

## Drill 2 (Fixes Problem 3b & Your Parentheses Bug on 5a!): Inside-Out RMSE & Helper Chaining

### Crucial Catch on What You Wrote for Problem 5a:
You noted that you wrote:
```python
return (np.mean(y - y_hat)**2)**0.5   # <-- CAREFUL! Parentheses bug!
```
Look closely at where `**2` is placed:
* In `np.mean(y - y_hat)**2`, Python calculates `np.mean(y - y_hat)` **first** (the average of the raw residuals), and **then** squares that average!
* But per **Problem 4b (`Class 34`)**, for any least-squares line, the average of the raw residuals `np.mean(y - y_hat)` is **always `0.0`**! If you take the mean before squaring, positive and negative residuals cancel out to zero!
* **The Fix — Read R-M-S-E from Inside Out (Right to Left):**
  1. **E**rror: `(y - y_hat)` *(or call `residuals(t, x, y)`)*
  2. **S**quared: `(y - y_hat) ** 2` *(Square each error **inside** `np.mean(...)`!)*
  3. **M**ean: `np.mean((y - y_hat) ** 2)`
  4. **R**oot: `np.sqrt(np.mean((y - y_hat) ** 2))` *(or `(np.mean((y - y_hat) ** 2)) ** 0.5`)*

### How to Solve Problem 3b (RMSE Using Provided Helpers Without `minimize`):
Because `residuals(t, x, y)` is **already given to you on the Cover Sheet** and returns `t.column(y) - fitted_values(t, x, y)` (which is `y - y_hat`!), you can compute the exact linear RMSE in **one or two lines**:
```python
res = residuals(t, x, y)
rmse = np.sqrt(np.mean(res ** 2))
```


In [ ]:
# ==============================================================================
# DRILL 2 WORKSPACE: Fill in every `...` below and run this cell!
# ==============================================================================

# First, run this demonstration to see why `(np.mean(y - y_hat)**2)**0.5` gives 0.0:
demo_res = residuals(demographics, 'College %', 'Median Income')
print("WRONG parentheses `(np.mean(res)**2)**0.5` gives:", (np.mean(demo_res) ** 2) ** 0.5)
print("RIGHT parentheses `(np.mean(res**2))**0.5` gives:", (np.mean(demo_res ** 2)) ** 0.5)

# 2A (Problem 3b Exact Replay): Using the provided `residuals(t, x, y)` helper function,
# compute the exact RMSE of predicting 'Median Income' from 'College %' in `demographics`:
res_demographics = residuals(..., ..., ...)
rmse_demographics = np.sqrt(np.mean(...))

# 2B (Second Practice): Using `residuals(t, x, y)`, compute the exact RMSE of predicting
# 'Birth Weight' from 'Gestational Days' in the `births` table in a SINGLE line:
rmse_births = ...

# --- AUTO-GRADER FOR DRILL 2 ---
_expected_dem_rmse = np.sqrt(np.mean(residuals(demographics, 'College %', 'Median Income') ** 2))
_expected_bwt_rmse = np.sqrt(np.mean(residuals(births, 'Gestational Days', 'Birth Weight') ** 2))
assert not isinstance(rmse_demographics, type(...)) and np.isclose(rmse_demographics, _expected_dem_rmse), "Check 2A: Square `res_demographics ** 2` INSIDE `np.mean(...)`!"
assert not isinstance(rmse_births, type(...)) and np.isclose(rmse_births, _expected_bwt_rmse), "Check 2B: np.sqrt(np.mean(residuals(births, 'Gestational Days', 'Birth Weight') ** 2))"
print(f"DRILL 2 PASSED! Demographics RMSE = {rmse_demographics:.2f}, Births RMSE = {rmse_births:.2f}")


## Drill 3 (Fixes Problem 5a & 5b): Writing `y_hat` (`estimate`) in Custom RMSE Functions & Using `coeffs.item(...)`

### Part 1 — How to Write `y_hat` (`estimate`) Inside Any RMSE Function (`Problem 5a`):
When `slope()` and `intercept()` don't apply (such as for a **quadratic curve** or **multiple regression**), we write a custom RMSE function and pass it to `minimize()`.
1. The arguments of your RMSE function are **only the unknown numbers (coefficients)** (`a, b`, or `a, b, c`, or `a, b, c, d`).
2. Pull the predictor column(s) `x = tbl.column(...)` and response column `y = tbl.column(...)` inside the function.
3. Write `y_hat` (`estimate`) using standard algebra (`** 2` for squared in Python, never `^ 2`!):
   * **Simple Line (`def lin_rmse(a, b)`):** `y_hat = a * x + b`
   * **Quadratic Curve (`def quad_rmse(a, b, c)`):** `y_hat = a * (x ** 2) + b * x + c`
   * **2-Predictor Multiple Regression (`def multi2_rmse(a, b, c)`):** `y_hat = a * w + b * x + c`
   * **3-Predictor Multiple Regression (`def multi3_rmse(a, b, c, d)`):** `y_hat = a * w + b * x + c * z + d`
4. Return `np.sqrt(np.mean((y - y_hat) ** 2))`!

### Part 2 — How to Predict Using `coeffs = minimize(...)` (`Problem 5b`):
`minimize(fn)` returns an **array of optimal coefficients in the exact same order** as your function arguments `(a, b, c)`:
* `coeffs.item(0)` is `a`
* `coeffs.item(1)` is `b`
* `coeffs.item(2)` is `c` (for a 2-predictor plane `a*w + b*x + c`, `c` is the **intercept**, so it stands alone at the end with no variable multiplied by it!)


In [ ]:
# ==============================================================================
# DRILL 3 WORKSPACE: Fill in every `...` below and run this cell!
# ==============================================================================

# 3A (Problem 5a Exact Replay): Complete `quad_rmse(a, b, c)` for predicting
# 'Median Income' (y) from 'College %' (x) using a parabola y_hat = a*(x**2) + b*x + c:
def quad_rmse(a, b, c):
    x = demographics.column('College %')
    y = demographics.column('Median Income')
    y_hat = ...
    return np.sqrt(np.mean((y - y_hat) ** 2))

best_quad = minimize(quad_rmse)

# 3B (Predicting with Quadratic Coefficients): Using `best_quad.item(...)`, predict
# the Median Income for a district with `College %` = 30:
pred_quad_30 = best_quad.item(0) * (30 ** 2) + best_quad.item(1) * ... + ...

# 3C (Problem 5b Exact Replay — 2-Predictor Multiple Regression):
# Complete `multi_rmse(a, b, c)` for predicting 'Median Income' (y) from
# 'College %' (w) and 'Clinton %' (x) using the plane y_hat = a * w + b * x + c:
def multi_rmse(a, b, c):
    w = demographics.column('College %')
    x = demographics.column('Clinton %')
    y = demographics.column('Median Income')
    y_hat = ...
    return np.sqrt(np.mean((y - y_hat) ** 2))

coeffs = minimize(multi_rmse)

# Now use `coeffs.item(0)`, `coeffs.item(1)`, and `coeffs.item(2)` to predict
# 'Median Income' for a district with College % = 25 (w=25) and Clinton % = 52 (x=52):
pred_multi_25_52 = ...

# --- AUTO-GRADER FOR DRILL 3 ---
_w = demographics.column('College %')
_x = demographics.column('Clinton %')
_y = demographics.column('Median Income')
assert not isinstance(best_quad, type(...)) and np.isclose(quad_rmse(1, 2, 3), np.sqrt(np.mean((_y - (1*(_w**2) + 2*_w + 3))**2))), "Check 3A: y_hat = a * (x ** 2) + b * x + c"
assert not isinstance(pred_quad_30, type(...)) and np.isclose(pred_quad_30, best_quad.item(0)*(30**2) + best_quad.item(1)*30 + best_quad.item(2)), "Check 3B: best_quad.item(0)*(30**2) + best_quad.item(1)*30 + best_quad.item(2)"
assert np.isclose(multi_rmse(10, 20, 30), np.sqrt(np.mean((_y - (10*_w + 20*_x + 30))**2))), "Check 3C multi_rmse: y_hat = a * w + b * x + c"
assert not isinstance(pred_multi_25_52, type(...)) and np.isclose(pred_multi_25_52, coeffs.item(0)*25 + coeffs.item(1)*52 + coeffs.item(2)), "Check 3C pred_multi_25_52: coeffs.item(0) * 25 + coeffs.item(1) * 52 + coeffs.item(2)"
print(f"DRILL 3 PASSED! Quadratic pred(30) = ${pred_quad_30:,.2f}, Multi pred(25, 52) = ${pred_multi_25_52:,.2f}")


## Drill 4 (Fixes Problem 6a & 6b): The 5-Step Bootstrap Simulation Skeleton

In `Class 34`, the professor explicitly stated that you **must** know how to write simulation loops and `percentile()` confidence intervals from memory. Every bootstrap function in Unit 7 follows the **exact same 5-line pattern**:

| Step | `bootstrap_prediction(t, x, y, new_x, repetitions)` (`Problem 6a`) | `bootstrap_slope(t, x, y, repetitions)` (`Problem 6b`) |
| :--- | :--- | :--- |
| **1. Empty Array** | `predictions = make_array()` | `slopes = make_array()` |
| **2. Loop Header** | `for i in np.arange(repetitions):` | `for i in np.arange(repetitions):` |
| **3. Resample Table** | `    resample = t.sample()` | `    resample = t.sample()` |
| **4. Compute Stat on `resample` & Append** | `    pred = prediction_at(resample, x, y, new_x)`<br>`    predictions = np.append(predictions, pred)` | `    b_slope = slope(resample, x, y)`<br>`    slopes = np.append(slopes, b_slope)` |
| **5. 95% Percentile Bounds** | `left = percentile(2.5, predictions)`<br>`right = percentile(97.5, predictions)`<br>`return make_array(left, right)` | `left = percentile(2.5, slopes)`<br>`right = percentile(97.5, slopes)`<br>`return make_array(left, right)` |

> **Watch Out:** Inside the loop, ALWAYS pass `resample` (never `t`!) into `prediction_at(resample, ...)` or `slope(resample, ...)`. If you pass `t`, you recompute the exact same sample slope 1,000 times!


In [ ]:
# ==============================================================================
# DRILL 4 WORKSPACE: Fill in every `...` below and run this cell!
# ==============================================================================

# 4A (Problem 6a Exact Replay): Complete `bootstrap_prediction` for a 95% CI:
def bootstrap_prediction(t, x, y, new_x, repetitions=500):
    predictions = make_array()
    for i in np.arange(...):
        resample = ...
        pred = prediction_at(..., ..., ..., ...)
        predictions = np.append(..., ...)
    left = percentile(..., predictions)
    right = percentile(..., predictions)
    return make_array(left, right)

# 4B (Problem 6b Exact Replay): Complete `bootstrap_slope` for a 95% CI:
def bootstrap_slope(t, x, y, repetitions=500):
    slopes = make_array()
    for i in np.arange(...):
        resample = ...
        b_slope = slope(..., ..., ...)
        slopes = np.append(..., ...)
    left = percentile(..., slopes)
    right = percentile(..., slopes)
    return make_array(left, right)

# 4C (90% Confidence Interval Bonus): What two percentiles bound the middle 90% of `slopes`?
# Hint: (100 - 90) / 2 = 5% in each tail!
left_pct_90 = ...
right_pct_90 = ...

# --- AUTO-GRADER FOR DRILL 4 ---
np.random.seed(42)
_ci_pred = bootstrap_prediction(births, 'Gestational Days', 'Birth Weight', 280, repetitions=200)
assert len(_ci_pred) == 2 and _ci_pred.item(0) < _ci_pred.item(1), "Check 4A: Make sure you resample with `t.sample()` (not `t`!) and use percentile(2.5, ...) and percentile(97.5, ...)"
assert 112.0 < _ci_pred.item(0) < _ci_pred.item(1) < 125.0, f"Check 4A bounds: Got {_ci_pred}, expected ~[117.5, 119.5]"

np.random.seed(42)
_ci_slope = bootstrap_slope(demographics, 'Clinton %', 'Median Income', repetitions=200)
assert len(_ci_slope) == 2 and _ci_slope.item(0) < _ci_slope.item(1), "Check 4B: Make sure you pass `resample` to `slope(resample, x, y)` and use 2.5 and 97.5 percentiles"
assert left_pct_90 == 5 and right_pct_90 == 95, "Check 4C: Middle 90% leaves 5% in the left tail (5) and 5% in the right tail (95)!"
print(f"DRILL 4 PASSED! 95% Prediction CI at 280d = {_ci_pred.round(2)}, 95% Slope CI = {_ci_slope.round(2)}")


## Drill 5 (Fixes Problems 3c(ii), 4a, 4b, and 6c(ii)): Interactive Conceptual Proofs & Short-Answer Templates

You also missed four conceptual/diagnostic subparts (**3c(ii), 4a, 4b, and 6c(ii)**). Run the code cell below to see the exact Python proof behind each one, and memorize the 1-sentence exam templates beneath it!

### 1-Sentence Exam Answer Templates to Memorize:
* **Problem 3c(ii) — Extrapolation Danger (`Class 34`):**
  > *"Predicting at $x = 95\%$ is **extrapolation** far outside the observed data range ($10\%$ to $65\%$), which is unreliable because we have no evidence that the linear relationship continues to hold outside our observed range."*
* **Problem 4a — The Two Visual Residual Plot Checks & Why Fanning Breaks CIs (`Class 32` & `Class 34`):**
  > *"On a residual plot, we check (1) **no nonlinear curvature** (Model 1 fails this because it shows a parabola/arch) and (2) **constant vertical spread / homoscedasticity** (Model 2 fails this because its residuals **fan out** as fitted values increase, which invalidates prediction confidence intervals that assume uniform error spread across the whole line)."*
* **Problem 4b — Why `correlation(Fitted, Residual) == 0` and `np.mean(residuals) == 0` Are Traps (`Class 34`):**
  > *"Mathematically, **every** ordinary least-squares regression line automatically has a mean residual of `0` (shown as `-1.42e-15` in Python due to 15-decimal floating-point rounding) and a correlation of `0` between fitted values and residuals—even when fit to a severe parabola! Therefore, numerical checks of mean or correlation cannot detect a bad linear fit; we must inspect the residual plot visually."*
* **Problem 6c(ii) — Statistical Significance vs. Practical Significance (`Class 34` Clinton % Example):**
  > *"Because `0` is **not** inside the 95% confidence interval `[$81.0, $270.5]`, we **reject $H_0$** and conclude the slope is **statistically significant** (non-zero in the population); however, it may not be **practically significant** because an $81/year difference in median household income per 1% vote share is tiny in real-world terms."*


In [ ]:
# Run this cell to see live Python proof of Problem 4b:
# Even when y = x^2 (a pure parabola!), fitting a straight line gives mean(residuals) = 0 and correlation(Fitted, Residual) = 0!
x_parab = np.linspace(-10, 10, 101)
y_parab = x_parab ** 2
parab_tbl = Table().with_columns('x', x_parab, 'y', y_parab)
parab_res_tbl = add_regression_to_table(parab_tbl, 'x', 'y')

print("Mean of residuals on a pure parabola:", np.mean(parab_res_tbl.column('Residual')))
print("Correlation between Fitted and Residual on a pure parabola:", correlation(parab_res_tbl, 'Fitted', 'Residual'))


## Complete Answer Key for Drills 1–4 (Only Peek If Stuck!)
```python
# --- DRILL 1 SOLUTIONS ---
area_left_85 = stats.norm.cdf(85, loc=100, scale=15)
cutoff_90th = stats.norm.ppf(0.90, loc=100, scale=15)
df_2b = min(n1, n2) - 1
pval_2b = 2 * (1 - stats.t.cdf(abs(t_stat), df_2b))
df_bonus = min(25, 30) - 1
pval_right = 1 - stats.t.cdf(1.85, df_bonus)

# --- DRILL 2 SOLUTIONS ---
res_demographics = residuals(demographics, 'College %', 'Median Income')
rmse_demographics = np.sqrt(np.mean(res_demographics ** 2))
rmse_births = np.sqrt(np.mean(residuals(births, 'Gestational Days', 'Birth Weight') ** 2))

# --- DRILL 3 SOLUTIONS ---
# Inside quad_rmse:
y_hat = a * (x ** 2) + b * x + c
# Predicting at x = 30:
pred_quad_30 = best_quad.item(0) * (30 ** 2) + best_quad.item(1) * 30 + best_quad.item(2)
# Inside multi_rmse:
y_hat = a * w + b * x + c
# Predicting at w = 25, x = 52:
pred_multi_25_52 = coeffs.item(0) * 25 + coeffs.item(1) * 52 + coeffs.item(2)

# --- DRILL 4 SOLUTIONS ---
def bootstrap_prediction(t, x, y, new_x, repetitions=500):
    predictions = make_array()
    for i in np.arange(repetitions):
        resample = t.sample()
        pred = prediction_at(resample, x, y, new_x)
        predictions = np.append(predictions, pred)
    left = percentile(2.5, predictions)
    right = percentile(97.5, predictions)
    return make_array(left, right)

def bootstrap_slope(t, x, y, repetitions=500):
    slopes = make_array()
    for i in np.arange(repetitions):
        resample = t.sample()
        b_slope = slope(resample, x, y)
        slopes = np.append(slopes, b_slope)
    left = percentile(2.5, slopes)
    right = percentile(97.5, slopes)
    return make_array(left, right)

left_pct_90 = 5
right_pct_90 = 95
```
